# BomberMan AI — A* + Danger Map + Minimax (fixed version)

**Fixes over the first version**
1. Bombs now **tick and explode** (timer is used, chain reactions included).
2. `ai_alive` / `enemy_alive` are **updated** when a blast hits a player, so win/lose branches work.
3. **A\* is used** inside the evaluation (escape cost from a blast zone).
4. The evaluation now counts **enemy danger**, so placing a trap bomb can pay off.
5. Distance term **keeps a useful distance** (3 cells) instead of rewarding closeness.
6. Warning zone uses **walking distance**, so it no longer leaks through walls.
7. `nearest_safe_cell` uses **real path distance** and returns `None` if the AI is already safe.

Game rule: one round = AI move, then enemy move; bombs tick after the enemy's move (a bomb explodes 3 rounds after it is placed).

In [ ]:
from dataclasses import dataclass, replace
from heapq import heappush, heappop
from collections import deque
from math import inf
from typing import List, Tuple, Dict, Set, Optional

Position = Tuple[int, int]
DIRECTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
MOVE_ACTIONS = {"UP": (-1, 0), "DOWN": (1, 0), "LEFT": (0, -1), "RIGHT": (0, 1), "WAIT": (0, 0)}
BOMB_TIMER = 3
IDEAL_DISTANCE = 3

## 1. State, blast cells, danger map

In [ ]:
@dataclass(frozen=True)
class Bomb:
    position: Position
    timer: int = BOMB_TIMER
    radius: int = 3

@dataclass
class GameState:
    grid: List[str]
    ai: Position
    enemy: Position
    bombs: Tuple[Bomb, ...] = ()
    ai_alive: bool = True
    enemy_alive: bool = True

    @property
    def rows(self): return len(self.grid)
    @property
    def cols(self): return len(self.grid[0])
    def is_inside(self, p): return 0 <= p[0] < self.rows and 0 <= p[1] < self.cols
    def is_wall(self, p): return self.grid[p[0]][p[1]] == "#"
    def is_blocked(self, p): return (not self.is_inside(p)) or self.is_wall(p)
    def neighbors(self, p):
        for dr, dc in DIRECTIONS:
            q = (p[0] + dr, p[1] + dc)
            if not self.is_blocked(q):
                yield q

In [ ]:
def blast_cells(state, bomb):
    """Cells hit by a bomb: 4 directions, stopped by walls."""
    affected = {bomb.position}
    for dr, dc in DIRECTIONS:
        for step in range(1, bomb.radius + 1):
            p = (bomb.position[0] + dr * step, bomb.position[1] + dc * step)
            if not state.is_inside(p) or state.is_wall(p):
                break
            affected.add(p)
    return affected

def blast_zone(state):
    zone = set()
    for b in state.bombs:
        zone |= blast_cells(state, b)
    return zone

def walking_distances(state, start, limit=None):
    """BFS walking distance (walls respected) from start."""
    dist = {start: 0}
    q = deque([start])
    while q:
        cur = q.popleft()
        if limit is not None and dist[cur] >= limit:
            continue
        for nxt in state.neighbors(cur):
            if nxt not in dist:
                dist[nxt] = dist[cur] + 1
                q.append(nxt)
    return dist

def build_danger_map(state):
    """Danger per walkable cell. Blast cells: 100 + urgency (shorter timer = worse).
    Warning zone uses WALKING distance, so danger does not leak through walls."""
    danger = {(r, c): 0 for r in range(state.rows) for c in range(state.cols)
              if state.grid[r][c] != "#"}
    for bomb in state.bombs:
        affected = blast_cells(state, bomb)
        urgency = 30 * max(0, BOMB_TIMER - bomb.timer)
        for p in affected:
            danger[p] += 100 + urgency
        for p, d in walking_distances(state, bomb.position, bomb.radius + 1).items():
            if p not in affected:
                danger[p] += max(0, 10 - 2 * d)
    return danger

def print_danger_map(state):
    danger = build_danger_map(state)
    for r in range(state.rows):
        print(" ".join("###" if state.grid[r][c] == "#" else f"{danger[(r, c)]:3d}"
                       for c in range(state.cols)))

## 2. A* and safe-cell search

In [ ]:
def heuristic(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def a_star(state, start, goal, danger=None):
    """A*: step cost = 1 + danger of the cell entered."""
    if danger is None:
        danger = build_danger_map(state)
    if state.is_blocked(start) or state.is_blocked(goal):
        return None
    heap = [(heuristic(start, goal), 0, start)]
    came_from, g_score = {}, {start: 0}
    while heap:
        _, g, cur = heappop(heap)
        if g > g_score.get(cur, inf):      # stale heap entry
            continue
        if cur == goal:
            path = [cur]
            while cur in came_from:
                cur = came_from[cur]
                path.append(cur)
            return path[::-1]
        for nxt in state.neighbors(cur):
            ng = g + 1 + danger.get(nxt, 0)
            if ng < g_score.get(nxt, inf):
                g_score[nxt] = ng
                came_from[nxt] = cur
                heappush(heap, (ng + heuristic(nxt, goal), ng, nxt))
    return None

def nearest_safe_cell(state, danger=None):
    """Nearest cell outside every blast zone (by walking distance).
    Returns None if the AI is already safe or no safe cell is reachable."""
    zone = blast_zone(state)
    if state.ai not in zone:
        return None
    if danger is None:
        danger = build_danger_map(state)
    dist = walking_distances(state, state.ai)
    cands = [p for p in dist if p not in zone]
    if not cands:
        return None
    return min(cands, key=lambda p: (dist[p], danger[p]))

## 3. Game dynamics (actions, bomb timers, explosions)

In [ ]:
def legal_actions(state, player="ai"):
    alive = state.ai_alive if player == "ai" else state.enemy_alive
    if not alive:
        return ["WAIT"]
    pos = state.ai if player == "ai" else state.enemy
    other = state.enemy if player == "ai" else state.ai
    actions = []
    for name, (dr, dc) in MOVE_ACTIONS.items():
        nxt = (pos[0] + dr, pos[1] + dc)
        if name == "WAIT" or (not state.is_blocked(nxt) and nxt != other):
            actions.append(name)
    if not any(b.position == pos for b in state.bombs):
        actions.append("BOMB")
    return actions

def tick(state):
    """End of a round: timers drop by 1; bombs at 0 explode (with chain reactions)
    and kill anyone standing in the blast."""
    bombs = [replace(b, timer=b.timer - 1) for b in state.bombs]
    exploding = [b for b in bombs if b.timer <= 0]
    remaining = [b for b in bombs if b.timer > 0]
    zone = set()
    while exploding:
        b = exploding.pop()
        cells = blast_cells(state, b)
        zone |= cells
        chain = [x for x in remaining if x.position in cells]
        remaining = [x for x in remaining if x.position not in cells]
        exploding.extend(chain)
    return GameState(state.grid, state.ai, state.enemy, tuple(remaining),
                     state.ai_alive and state.ai not in zone,
                     state.enemy_alive and state.enemy not in zone)

def apply_action(state, action, player="ai"):
    """Apply one player's move. A round = AI move + enemy move; bombs tick after the enemy's move."""
    ai, enemy, bombs = state.ai, state.enemy, list(state.bombs)
    alive = state.ai_alive if player == "ai" else state.enemy_alive
    if alive:
        if action == "BOMB":
            bombs.append(Bomb(ai if player == "ai" else enemy))
        else:
            dr, dc = MOVE_ACTIONS[action]
            if player == "ai":
                new = (ai[0] + dr, ai[1] + dc)
                if not state.is_blocked(new) and new != enemy:
                    ai = new
            else:
                new = (enemy[0] + dr, enemy[1] + dc)
                if not state.is_blocked(new) and new != ai:
                    enemy = new
    new_state = GameState(state.grid, ai, enemy, tuple(bombs), state.ai_alive, state.enemy_alive)
    return tick(new_state) if player == "enemy" else new_state

## 4. Evaluation and Minimax with Alpha-Beta
Because a bomb needs 3 rounds (6 plies) to explode, the default depth is 6.

In [ ]:
def evaluate_state(state):
    """Higher = better for the AI."""
    if not state.ai_alive and not state.enemy_alive:
        return 0
    if not state.ai_alive:
        return -10000
    if not state.enemy_alive:
        return 10000
    danger = build_danger_map(state)
    score = 1000
    score -= 20 * danger.get(state.ai, 0)                      # own danger
    score += 15 * danger.get(state.enemy, 0)                   # enemy danger (reward trapping)
    score -= 2 * abs(heuristic(state.ai, state.enemy) - IDEAL_DISTANCE)
    score += 3 * len([a for a in legal_actions(state, "ai") if a not in ("WAIT", "BOMB")])
    safe = nearest_safe_cell(state, danger)                    # A* escape cost
    if state.ai in blast_zone(state):
        path = a_star(state, state.ai, safe, danger) if safe else None
        score -= 15 * (len(path) - 1) if path else 500
    return score

def minimax(state, depth, maximizing, alpha=-inf, beta=inf):
    if depth == 0 or not state.ai_alive or not state.enemy_alive:
        s = evaluate_state(state)
        if s >= 10000: s += depth       # win sooner
        elif s <= -10000: s -= depth    # lose later
        return s, None
    player = "ai" if maximizing else "enemy"
    best_action = None
    best = -inf if maximizing else inf
    for action in legal_actions(state, player):
        score, _ = minimax(apply_action(state, action, player), depth - 1, not maximizing, alpha, beta)
        if (maximizing and score > best) or (not maximizing and score < best):
            best, best_action = score, action
        if maximizing: alpha = max(alpha, best)
        else: beta = min(beta, best)
        if beta <= alpha:
            break
    return best, best_action

def choose_ai_action(state, depth=6):
    score, action = minimax(state, depth, True)
    return action, score

## 5. Test suite

### Test 1 — A* avoids the dangerous route

In [ ]:
grid1 = ["#########","#.......#","#.#####.#","#.......#","#########"]
s = GameState(grid1, (1,1), (1,7), (Bomb((1,4), timer=3, radius=2),))
print_danger_map(s)
path = a_star(s, s.ai, (1,7)); print(path)
assert path is not None and not (set(path) & blast_zone(s)), "path enters blast zone"
assert len(path) - 1 == 10
print("PASS")

### Test 2 — Nearest safe cell (real path distance)

In [ ]:
grid2 = ["#########","#.......#","#.......#","#.......#","#########"]
s = GameState(grid2, (2,2), (2,6), (Bomb((2,4), radius=2), Bomb((1,6), radius=2)))
safe = nearest_safe_cell(s); path = a_star(s, s.ai, safe)
print("safe:", safe, "path:", path)
assert safe not in blast_zone(s) and path[0] == s.ai and path[-1] == safe
print("PASS")

### Test 3 — Already safe means no escape needed

In [ ]:
s = GameState(grid2, (1,1), (3,7), (Bomb((2,4), radius=1),))
assert nearest_safe_cell(s) is None
print("PASS")

### Test 4 — Bomb timer: explosion kills a player who stays

In [ ]:
s = GameState(grid2, (2,4), (3,7), (Bomb((2,4), timer=1, radius=3),))
n = apply_action(apply_action(s, "WAIT", "ai"), "WAIT", "enemy")
print("ai_alive:", n.ai_alive, "bombs left:", len(n.bombs))
assert not n.ai_alive and n.enemy_alive and n.bombs == ()
print("PASS")

### Test 5 — Chain reaction

In [ ]:
s = GameState(grid2, (1,1), (2,7), (Bomb((2,4), timer=1, radius=1), Bomb((2,5), timer=3, radius=3)))
n = tick(s)
print("bombs left:", n.bombs, "| enemy alive:", n.enemy_alive)
assert n.bombs == () and not n.enemy_alive   # only the chained bomb at (2,5) can reach the enemy at (2,7)
print("PASS")

### Test 6 — Walls stop blasts and warnings

In [ ]:
g = ["#######","#.#...#","#.#...#","#######"]
s = GameState(g, (1,1), (1,5), (Bomb((1,3), radius=1),))
d = build_danger_map(s)
print("danger behind wall:", d[(1,1)])
assert d[(1,1)] == 0 and (1,2) not in blast_zone(s)
print("PASS")

### Test 7 — Timer-aware danger

In [ ]:
s1 = GameState(grid2, (1,1), (3,7), (Bomb((2,4), timer=3),))
s2 = GameState(grid2, (1,1), (3,7), (Bomb((2,4), timer=1),))
d1, d2 = build_danger_map(s1)[(2,4)], build_danger_map(s2)[(2,4)]
print(d1, "<", d2); assert d1 < d2
print("PASS")

### Test 8 — AI escapes an imminent bomb

In [ ]:
s = GameState(grid2, (2,3), (2,7), (Bomb((2,4), timer=1, radius=2),))
act, sc = choose_ai_action(s, 4)
n = apply_action(apply_action(s, act, "ai"), "WAIT", "enemy")
print("action:", act, "| alive after explosion:", n.ai_alive)
assert n.ai_alive
print("PASS")

### Test 9 — AI traps the enemy in a dead end with a bomb

In [ ]:
corridor = ["#######","#.....#","#####.#","#####.#","#######"]
s = GameState(corridor, (1,4), (1,1))
act, sc = choose_ai_action(s, 6)
print("action:", act, "| score:", sc)
assert act == "BOMB" and sc >= 10000
print("PASS")

### Test 10 — Open board: legal actions and a decision

In [ ]:
grid4 = ["#######","#.....#","#.....#","#.....#","#######"]
s = GameState(grid4, (2,2), (2,4))
print(legal_actions(s, "ai"))
assert set(legal_actions(s, "ai")) == {"UP","DOWN","LEFT","RIGHT","WAIT","BOMB"}
print(choose_ai_action(s, 4)); print("PASS")

### Test 11 — Alpha-beta equals plain minimax (random states)

In [ ]:
import random
def plain(state, depth, mx):
    if depth == 0 or not state.ai_alive or not state.enemy_alive:
        s = evaluate_state(state)
        return s + depth if s >= 10000 else s - depth if s <= -10000 else s
    p = "ai" if mx else "enemy"
    v = [plain(apply_action(state, a, p), depth-1, not mx) for a in legal_actions(state, p)]
    return max(v) if mx else min(v)
random.seed(1); free = [(r,c) for r in range(1,4) for c in range(1,8)]
for _ in range(20):
    a_, e_ = random.sample(free, 2)
    bs = tuple(Bomb(random.choice(free), random.randint(1,3), random.randint(1,3)) for _ in range(random.randint(0,2)))
    s = GameState(grid2, a_, e_, bs)
    assert minimax(s, 4, True)[0] == plain(s, 4, True)
print("PASS (20 random states)")

### Test 12 — Evaluation uses A* and keeps useful distance

In [ ]:
calls = []; _orig = a_star
a_star = lambda *a, **k: (calls.append(1), _orig(*a, **k))[1]
evaluate_state(GameState(grid2, (2,3), (2,7), (Bomb((2,4), timer=1, radius=2),)))
a_star = _orig
print("A* calls inside evaluation:", len(calls)); assert calls
near = evaluate_state(GameState(grid2, (2,2), (2,3))); good = evaluate_state(GameState(grid2, (2,2), (2,5)))
print("adjacent:", near, "| distance 3:", good); assert good > near
print("PASS")

### Test 13 — A* edge cases

In [ ]:
s = GameState(grid2, (1,1), (3,7))
assert a_star(s, (1,1), (0,0)) is None and a_star(s, (1,1), (1,1)) == [(1,1)]
assert len(a_star(s, (1,1), (3,7))) - 1 == 8
g = ["#####","#.#.#","#####"]; assert a_star(GameState(g,(1,1),(1,3)), (1,1), (1,3)) is None
print("PASS")

### Test 14 — Self-play vs a random enemy (200 rounds max, 30 games)

In [ ]:
import random, time
def random_enemy(s, rng):
    acts = legal_actions(s, "enemy"); return rng.choice(acts)
rng = random.Random(7); res = {"AI wins":0, "AI died":0, "draw/timeout":0}; t0 = time.time()
for g_ in range(30):
    s = GameState(grid2, (1,1), (3,7))
    for _ in range(60):
        a, _ = choose_ai_action(s, 6); s = apply_action(s, a, "ai")
        if not s.ai_alive or not s.enemy_alive: break
        s = apply_action(s, random_enemy(s, rng), "enemy")
        if not s.ai_alive or not s.enemy_alive: break
    res["AI wins" if s.ai_alive and not s.enemy_alive else "AI died" if not s.ai_alive and s.enemy_alive else "draw/timeout"] += 1
print(res, f"({time.time()-t0:.1f}s)")

## 6. Complexity
**A\***: O((V + E) log V) with a binary heap (stale heap entries are now skipped).
**Minimax**: O(b^d), b ≤ 6 actions (4 moves, WAIT, BOMB), d = 6. Alpha-beta best case O(b^(d/2)).
**Architecture**: Danger Map → A* escape cost inside the evaluation → Minimax chooses the action.